# PA3 · Task 1 · Training (Full FT, L2-SP, LoRA-4, LoRA-16, OPLoRA-16)

**What this notebook does.** Prepares the 40,000 MetaMathQA examples, runs a pre-flight check of the real model in FP16, then trains all five methods in **one session**, one process per GPU, and checks every run against the manual.

| GPU 0 | GPU 1 |
|---|---|
| `full` (Full FT), then `l2sp` (Full FT + L2-SP) | `lora4`, then `lora16`, then `oplora` |

**Time:** estimated 6–9 h on Kaggle **GPU T4 ×2** (Kaggle's limit is 12 h). Each run's log prints a real ETA after its first 25 updates.
**Checkpoints and the 12-hour limit:**
- Every run saves resumable checkpoints after 25 / 50 / 75 / 100% of its updates (the manual's schedule), plus an extra one whenever an hour passes without a save.
- Anything still training after `STOP_AFTER_HOURS` (11 h) saves a checkpoint and pauses, so the notebook finishes before Kaggle's 12-hour cut-off and the output is kept.
- If anything paused or crashed, run this notebook again with that output attached: finished runs are copied over, the others resume from their newest checkpoint with bit-identical results. Repeat until the last cell says all five are finished.

**Settings:** Accelerator **GPU T4 ×2**, Internet **On**.
**Attach:** `pa3-task1`. To continue a paused or crashed session, or to retrain only the methods in `RERUN` (cell S2), also attach the earlier session's output (*Add Input → Your Work → this notebook's output*): finished runs not in `RERUN` are copied over.

**Run with "Save Version → Save & Run All"** so the output is kept even if you close the tab.

**How to read the output:** every check prints `PASS / FAIL / OBSERVE`, then the **expected** and **actual** lines. `OBSERVE` is a measurement for the report, not a test.

In [ ]:
# S1. Find the Task 1 code (attached Kaggle dataset, or a clone of your GitHub repo), copy it to /tmp, check the GPUs.
# All GPU work runs in separate processes, so this notebook itself never holds GPU memory.
import os, sys, glob, json, time, shutil, subprocess
from pathlib import Path
SESSION_START = time.time()     # Kaggle's 12-hour limit counts from about now

REPO_URL = ""        # optional: your public GitHub repo URL, used only if no code dataset is attached
REQUIRED_CODE_VERSION = "2026.10.30.2" # this notebook needs this code version or newer
CODE = Path("/tmp/pa3_task1_code")

def code_version(root):
    import re
    try:
        m = re.search(r'CODE_VERSION\s*=\s*"([^"]+)"', (Path(root) / "common" / "__init__.py").read_text())
        return m.group(1) if m else "0 (no version stamp: older than 2026.10.30)"
    except FileNotFoundError:
        return "0 (no version stamp: older than 2026.10.30)"

# every attached copy of the code, newest version first (a dataset can hold several copies after re-uploads)
hits = sorted({Path(p).parents[1] for p in glob.glob("/kaggle/input/**/task1/t1_train.py", recursive=True)},
              key=code_version, reverse=True)
for h in hits:
    print(f"found code copy: {h}  (version {code_version(h)})")
if CODE.exists():
    shutil.rmtree(CODE)
if hits:
    if code_version(hits[0]) < REQUIRED_CODE_VERSION:
        raise SystemExit(f"The newest attached code is version {code_version(hits[0])}, but this notebook needs "
                         f"{REQUIRED_CODE_VERSION}. Upload the latest pa3-task1.zip as a new dataset version and re-attach it.")
    shutil.copytree(hits[0], CODE)
    print("using:", hits[0])
elif REPO_URL:
    subprocess.run(["git", "clone", "-q", REPO_URL, str(CODE)], check=True)
    print("code cloned from", REPO_URL)
else:
    raise SystemExit("Attach the pa3-task1 dataset (or set REPO_URL) and re-run this cell.")
sys.path.insert(0, str(CODE))
os.chdir(CODE)
CODE_COPIES = [str(h) for h in hits]
def inputs(pattern):
    """Files matching `pattern` in the ATTACHED OUTPUTS only: earlier runs and results are always read from the original
    notebook output you attached, never from a copy inside the code dataset."""
    return [p for p in glob.glob(pattern, recursive=True) if not any(p.startswith(c + "/") for c in CODE_COPIES)]
from common.utils import Checks, human_bytes, dir_size_bytes, read_json, read_jsonl
CHK = Checks()
CHK.check(code_version(CODE) >= REQUIRED_CODE_VERSION, "code version in use", f">= {REQUIRED_CODE_VERSION}", code_version(CODE))
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
    GPUS = [l.strip() for l in smi.stdout.strip().splitlines() if l.strip()]
except FileNotFoundError:
    GPUS = []
CHK.check(len(GPUS) >= 1, "GPU visible", "GPU T4 x2 (Settings -> Accelerator)", GPUS or "no GPU: is the accelerator switched on?")
import importlib.metadata as md
vers = {p: md.version(p) for p in ["torch", "transformers", "datasets"]}
CHK.observe("library versions", vers)
tv = tuple(int(x) for x in vers["transformers"].split(".")[:2])
if tv < (4, 51):     # Qwen3 support arrived in transformers 4.51
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"], check=True)
    print("upgraded transformers: restart the kernel and run again")

In [ ]:
# S2. CONFIG. One session: GPU 0 runs the two full-model methods, GPU 1 the three adapter methods.
PLAN = {0: ["full", "l2sp"], 1: ["lora4", "lora16", "oplora"]}
STOP_AFTER_HOURS = 11.0  # runs still training at this point pause themselves (Kaggle stops sessions at 12 h)
RERUN = ["oplora"]       # train these again from scratch even if a FINISHED copy is attached (a PAUSED one still resumes).
                         # ["oplora"]: retrain OPLoRA with the re-orthonormalised bases, reuse the other four. [] = reuse everything
WORK = Path("/kaggle/working")
DATA = WORK / "t1_data"
RUNS = WORK / "t1_runs"
REF_STORAGE = "auto"     # decision D3, where L2-SP keeps theta0: auto | cpu_bf16 | gpu_bf16 | gpu_fp32 (auto = pinned CPU on a 16 GB T4)
STOP_AT = SESSION_START + STOP_AFTER_HOURS * 3600
if len(GPUS) < 2:        # one GPU: everything runs one after another (expect to need a second session)
    PLAN = {0: [m for g in sorted(PLAN) for m in PLAN[g]]}
CHK.observe("session plan", PLAN, note=f"{len(GPUS)} GPU(s) visible; runs pause at {time.strftime('%H:%M', time.localtime(STOP_AT))} (container clock)")

In [ ]:
# S3. Data: reuse an attached t1_data if present, otherwise build it on the CPU (about 5 min).
prev = [Path(p).parent for p in inputs("/kaggle/input/**/t1_data/t1_train_tokens.npz")]
if (DATA / "t1_train_tokens.npz").exists():
    print("data already in the working directory")
elif prev:
    shutil.copytree(prev[0], DATA)
    print("copied prepared data from", prev[0])
else:
    from task1.t1_data import prepare
    prepare(str(DATA), CHK)
ids = read_json(DATA / "t1_train_ids.json")
CHK.check(len(ids["source_row_indices"]) == 40000, "40,000 training examples recorded", 40000, len(ids["source_row_indices"]))
CHK.observe("exclusion counts", ids["exclusion_counts"])
probe = read_json(DATA / "t1_probe_set.json")
CHK.check(len(probe["gsm8k"]) == 32 and len(probe["hellaswag"]) == 32, "probe set saved (32 GSM8K + 32 HellaSwag)", "32 + 32",
          f"{len(probe['gsm8k'])} + {len(probe['hellaswag'])}")

In [ ]:
# S4. Pre-flight on the real model (separate process): does FP16 behave like FP32 on a few prompts?
#     Training and evaluation must use FP16 on the T4, so a broken FP16 forward pass would invalidate everything.
code = r"""
import torch, json
from transformers import AutoTokenizer
from common.utils import load_causal_lm
from task1.t1_data import build_prompt
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
texts = [build_prompt(tok, "Q: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether?\nA:"),
         "Roof shingle removal: A man is sitting on a roof. He"]
out = {}
for name, dt in [("fp32", torch.float32), ("fp16", torch.float16)]:
    m = load_causal_lm("Qwen/Qwen3-0.6B", dt).cuda().eval()
    with torch.no_grad():
        out[name] = [m(**tok(t, return_tensors="pt", add_special_tokens=False).to("cuda")).logits.float().cpu() for t in texts]
    del m; torch.cuda.empty_cache()
fin = all(torch.isfinite(l).all().item() for l in out["fp16"])
agree = sum((a.argmax(-1) == b.argmax(-1)).float().mean().item() for a, b in zip(out["fp32"], out["fp16"])) / 2
diff = max((a - b).abs().max().item() for a, b in zip(out["fp32"], out["fp16"]))
print(json.dumps({"finite": fin, "argmax_agreement": agree, "max_abs_logit_diff": diff, "prompt": texts[0]}))
"""
r = subprocess.run([sys.executable, "-c", code], cwd=CODE, capture_output=True, text=True, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
res = json.loads([l for l in r.stdout.splitlines() if l.startswith("{")][-1]) if r.returncode == 0 else None
if res is None:
    print(r.stderr[-3000:])
CHK.check(res is not None and res["finite"], "FP16 forward pass produces finite logits", True, None if res is None else res["finite"])
if res:
    CHK.check(res["argmax_agreement"] > 0.95, "FP16 and FP32 pick the same next token almost everywhere", "> 0.95", round(res["argmax_agreement"], 4))
    CHK.observe("largest FP16 vs FP32 logit difference", round(res["max_abs_logit_diff"], 4), note="small values are normal FP16 rounding")
    print("chat-formatted GSM8K prompt as the model sees it:\n" + repr(res["prompt"]))

In [ ]:
# S5. Launch: one background chain of runs per GPU. Logs go to t1_runs/<method>.log
#     Runs finished in an earlier session are copied over from the attached output; paused ones resume.
RUNS.mkdir(parents=True, exist_ok=True)
def attached(m, marker):
    hits = sorted(inputs(f"/kaggle/input/**/t1_runs/{m}/{marker}"))
    return Path(hits[-1]).parent if hits else None
procs = {}
for gpu, methods in PLAN.items():
    cmds = []
    for m in methods:
        if (RUNS / m / "DONE").exists():
            print(f"{m}: already DONE"); continue
        prev_done = attached(m, "DONE")
        if prev_done is not None and m not in RERUN:
            shutil.copytree(prev_done, RUNS / m, dirs_exist_ok=True)
            print(f"{m}: finished in an earlier session, copied from {prev_done}"); continue
        if prev_done is not None:
            print(f"{m}: a finished copy is attached but {m} is in RERUN -> training it again from scratch")
        resume = sorted([p for p in inputs(f"/kaggle/input/**/t1_runs/{m}/recovery_u*/state.pt")
                         if not (Path(p).parents[1] / "DONE").exists()],
                        key=lambda p: int(Path(p).parent.name.split("_u")[1]))
        extra = f" --resume_from {Path(resume[-1]).parent}" if resume and not (RUNS / m).exists() else ""
        if extra: print(f"{m}: will resume from {Path(resume[-1]).parent}")
        cmds.append(f"python -u -m task1.t1_train --method {m} --data {DATA} --out {RUNS} --ref_storage {REF_STORAGE} "
                    f"--stop_at {STOP_AT:.0f}{extra} >> {RUNS}/{m}.log 2>&1")
    if not cmds: continue
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "TOKENIZERS_PARALLELISM": "false"}
    procs[gpu] = subprocess.Popen(["bash", "-c", " ; ".join(cmds)], cwd=CODE, env=env)   # ';' = a crash does not block the next run
    print(f"GPU {gpu}: started {methods}")

In [ ]:
# S6. Monitor until every queue finishes: checks every 30 s, prints the newest progress line of each run every 10 min.
def tail(path, n=1):
    try:
        lines = [l for l in open(path) if l.strip() and "it/s]" not in l]
        return "".join(lines[-n:]).strip()
    except FileNotFoundError:
        return "(not started)"
t_start, t_print = time.time(), time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(30)
    if time.time() - t_print >= 600:
        t_print = time.time()
        print(f"--- {(time.time() - t_start) / 3600:.2f} h elapsed (session {(time.time() - SESSION_START) / 3600:.2f} h)")
        for gpu, methods in PLAN.items():
            for m in methods:
                print(f"  [{m}] {tail(RUNS / f'{m}.log')}")
print(f"all queues finished after {(time.time() - t_start) / 3600:.2f} h")
for gpu, methods in PLAN.items():
    for m in methods:
        r = RUNS / m
        if (r / "DONE").exists():
            CHK.check(True, f"{m}: finished")
        elif (r / "PAUSED").exists():
            CHK.observe(f"{m}: paused at the time limit", f"update {read_json(r / 'PAUSED')['update']} / 1250",
                        note="run this notebook again with this version's output attached to continue")
        else:
            CHK.check(False, f"{m}: finished", "DONE", "stopped without finishing: last log lines below")
            print(f"==== last lines of {m}.log ====\n" + tail(RUNS / f"{m}.log", 40))

In [ ]:
# S7. Intended vs actual for every run in this session.
EXPECTED_PARAMS = {"full": 596_049_920, "l2sp": 596_049_920, "lora4": 2_523_136, "lora16": 10_092_544, "oplora": 10_092_544}
STATUS = {}
for m in [m for g in sorted(PLAN) for m in PLAN[g]]:
    r = RUNS / m
    STATUS[m] = ("DONE" if (r / "DONE").exists() else
                 f"PAUSED at update {read_json(r / 'PAUSED')['update']} / 1250" if (r / "PAUSED").exists() else
                 "not started" if not r.exists() else f"STOPPED (crashed: see t1_runs/{m}.log)")
    print(f"{m:7s} {STATUS[m]}")
for gpu, methods in PLAN.items():
    for m in methods:
        r = RUNS / m
        if STATUS[m] != "DONE":
            continue
        f = read_json(r / "final_summary.json"); log = read_jsonl(r / "train_log.jsonl"); ck = read_jsonl(r / "checkpoints.jsonl")
        cfg = read_json(r / "config.json")
        print(f"\n======== {m} ========")
        CHK.check(f["updates"] == 1250 and len(log) == 1250, f"{m}: 1,250 optimizer updates logged", 1250, (f["updates"], len(log)))
        CHK.check([c["update"] for c in ck] == [312, 625, 937, 1250], f"{m}: recovery checkpoints after 9,984 / 20,000 / 29,984 / 40,000 examples",
                  [312, 625, 937, 1250], [c["update"] for c in ck])
        CHK.check(f["trainable_params"] == EXPECTED_PARAMS[m], f"{m}: trainable parameter count", f"{EXPECTED_PARAMS[m]:,}", f"{f['trainable_params']:,}")
        CHK.check(abs(max(x["lr"] for x in log) - cfg["lr"]) < 1e-12 and log[0]["lr"] == 0.0 and log[-1]["lr"] < 0.01 * cfg["lr"],
                  f"{m}: LR schedule (0 at update 1, peak {cfg['lr']:g} after warm-up, ~0 at the end)", cfg["lr"],
                  (log[0]["lr"], max(x["lr"] for x in log), log[-1]["lr"]))
        CHK.observe(f"{m}: skipped updates (FP16 overflow)", f["skipped_updates"], note="a handful early on is normal for dynamic loss scaling")
        CHK.observe(f"{m}: SFT loss, mean of first 25 vs last 25 updates",
                    (round(sum(x['sft_loss'] for x in log[:25]) / 25, 4), round(sum(x['sft_loss'] for x in log[-25:]) / 25, 4)))
        CHK.observe(f"{m}: training time / peak GPU memory", f"{f['train_seconds_measured'] / 3600:.2f} h / {f['peak_allocated_gb']:.2f} GB")
        if m in ("full", "l2sp"):
            CHK.check((r / "export_final").exists(), f"{m}: FP16 export for evaluation", "export_final/", "present" if (r / "export_final").exists() else "missing")
            CHK.check(f["overhead"].get("theta0_bf16_exact", False), f"{m}: BF16 copy of theta0 is bit-exact", True, f["overhead"].get("theta0_bf16_exact"))
            CHK.observe(f"{m}: ||theta - theta0|| at each checkpoint", {c["update"]: round(c["displacement_l2"], 4) for c in ck})
        if m == "full":
            CHK.check((r / "export_mid").exists() and (r / "diag_down_proj_mid.pt").exists(), "full: midpoint export + diagnostic slices",
                      "present", "present" if (r / "export_mid").exists() else "missing")
        if m == "l2sp":
            last = log[-1]
            CHK.observe("l2sp: final Omega, lambda*Omega, SFT loss", (round(last["omega"], 3), round(last["lam_omega"], 5), round(last["sft_loss"], 4)),
                        note="compare lambda*Omega with the SFT loss: how strong was the penalty?")
            CHK.observe("l2sp: theta0 storage and overhead", {k: f["overhead"].get(k) for k in
                        ["reference_storage", "reference_storage_bytes", "reference_transfer_seconds_per_update"]} | {"penalty_seconds": f.get("penalty_seconds")})
        if m == "oplora":
            CHK.check(f["oplora_worst_rel_UkT_dW"] < 1e-4 and f["oplora_worst_rel_dW_Vk"] < 1e-4,
                      "oplora: constraints hold on all 196 matrices (||Uk^T dW|| / ||dW||, ||dW Vk|| / ||dW||)", "< 1e-4",
                      (f"{f['oplora_worst_rel_UkT_dW']:.2e}", f"{f['oplora_worst_rel_dW_Vk']:.2e}"))
            a = read_json(r / "injection.json")
            g = glob.glob(str(RUNS / "lora16" / "injection.json")) + inputs("/kaggle/input/**/t1_runs/lora16/injection.json")
            if g:
                b = read_json(g[0])
                CHK.check(a["init_A_sha256"] == b["init_A_sha256"], "oplora and lora16 started from identical A matrices", b["init_A_sha256"][:16], a["init_A_sha256"][:16])
            else:
                print("(attach session A's output to check that OPLoRA and LoRA-16 started from the same A)")
            CHK.observe("oplora overhead", {k: f["overhead"].get(k) for k in ["basis_svd_seconds", "basis_storage_bytes", "export_seconds"]})
            CHK.check(a.get("basis_method") is not None and "re-orthonormalisation" in a["basis_method"],
                      "oplora: bases re-orthonormalised once after the FP32 SVD", "re-orthonormalised", a.get("basis_method"))

In [ ]:
# S8. Disk: Kaggle keeps at most 20 GB of /kaggle/working. Show what is being saved.
for p in sorted(RUNS.iterdir()):
    if p.is_dir():
        print(f"{p.name:10s} {human_bytes(dir_size_bytes(p))}")
        for q in sorted(p.iterdir()):
            print(f"    {q.name:32s} {human_bytes(dir_size_bytes(q))}")
total = dir_size_bytes(WORK)
CHK.check(total < 19.5e9, "working directory fits Kaggle's 20 GB output limit", "< 19.5 GB", human_bytes(total))
CHK.summary(); CHK.save(WORK / f"t1_train_checks_{time.strftime('%Y%m%d_%H%M')}.json")
if all(v == "DONE" for v in STATUS.values()):
    print("\nNEXT: all five runs are finished. Open t1_eval and attach pa3-task1 + THIS version's output.")
else:
    print("\nNEXT: not everything finished:", {m: v for m, v in STATUS.items() if v != "DONE"},
          "\nRun this notebook again (Save & Run All) with THIS version's output attached; it continues where it stopped.")